In [ ]:
!pip install langchain chromadb pypdf langchain-google-genai langchain-community langchain-chroma groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.4 MB/s eta 0:00:00


In [ ]:
import os

for f in os.listdir("/content/"):
    print(f)

.config
chroma_db_backup (1).zip
chroma_db
sample_data


In [ ]:
import zipfile
import os

try:
    with zipfile.ZipFile("/content/chroma_db_backup (1).zip", "r") as zip_ref:
        zip_ref.extractall("/content/")

    if os.path.exists("/content/chroma_db"):
        print("✅ chroma_db restored successfully")
    else:
        raise FileNotFoundError("chroma_db folder not found after unzip")

except FileNotFoundError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to unzip: {e}")

✅ chroma_db restored successfully


In [ ]:
from google.colab import userdata
import os

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
    GROQ_KEY = userdata.get("GROQ_KEY")

    if not os.environ["GOOGLE_API_KEY"]:
        raise ValueError("GEMINI_KEY is empty — check Colab Secrets")
    if not GROQ_KEY:
        raise ValueError("GROQ_KEY is empty — check Colab Secrets")

    print("✅ API keys loaded successfully")

except KeyError as e:
    print(f"❌ Secret not found: {e} — add it in the 🔑 Secrets panel")
except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ API keys loaded successfully


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    print("✅ Embeddings model loaded successfully")

except Exception as e:
    print(f"❌ Failed to load embeddings model: {e}")

✅ Embeddings model loaded successfully


In [ ]:
from langchain_chroma import Chroma

try:
    vectorstore = Chroma(
        persist_directory="/content/chroma_db",
        embedding_function=embeddings
    )

    count = vectorstore._collection.count()
    if count == 0:
        raise ValueError("ChromaDB is empty — check your uploaded zip file")

    print(f"✅ Vector store loaded — {count} chunks available")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to load ChromaDB: {e}")

✅ Vector store loaded — 297 chunks available


In [ ]:
from groq import Groq

try:
    client = Groq(api_key=GROQ_KEY)
    print("✅ Groq client initialized successfully")

except Exception as e:
    print(f"❌ Failed to initialize Groq client: {e}")

✅ Groq client initialized successfully


In [ ]:
def ask_question(question):
    try:
        if not question or not question.strip():
            raise ValueError("Question cannot be empty")

        # Step 1: Find the 5 most relevant chunks from ChromaDB
        retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        relevant_chunks = retriever.invoke(question)

        if not relevant_chunks:
            raise ValueError("No relevant chunks found")

        # Step 2: Combine chunks into one context block
        context = "\n\n".join([doc.page_content for doc in relevant_chunks])

        # Step 3: Build the prompt and send to Groq
        prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}
Answer:"""

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        return {
            "answer": response.choices[0].message.content,
            "sources": [doc.page_content for doc in relevant_chunks]
        }

    except ValueError as e:
        return {"answer": f"❌ Input error: {e}", "sources": []}
    except Exception as e:
        return {"answer": f"❌ Failed to get answer: {e}", "sources": []}

print("✅ ask_question function ready")

✅ ask_question function ready


In [ ]:
try:
    result = ask_question("What is a neural network?")

    print("🤖 Answer:")
    print(result["answer"])

    print("\n--- Retrieved Chunks ---")
    for i, chunk in enumerate(result["sources"], 1):
        print(f"\nChunk {i}:\n{chunk[:300]}...")

except Exception as e:
    print(f"❌ Unexpected error: {e}")

🤖 Answer:
A neural network is a group of interconnected units called neurons that send signals to one another. Neurons can be either biological cells or mathematical models.

--- Retrieved Chunks ---

Chunk 1:
A neural network is a group of interconnected units called neurons that send signals to one another. Neurons can be either biological cells or mathematical models. While individual neurons are simple, many of them together in a network can perform complex tasks. There are two main types of neural ne...

Chunk 2:
In machine learning
In machine learning, a neural network is an artificial mathematical model used to approximate nonlinear functions. While early artificial neural networks were physical machines, today they are almost always implemented in software.
Neurons in an artificial neural network are usua...

Chunk 3:
Neural networks
Artificial neural networks (ANNs) or connectionist systems are computing systems inspired by the biological neural networks that constitute animal